In [ ]:
# /opt/conda/envs/aneris-dna/lib/R/
# =====
# Hot to publish data to GBIF: https://www.gbif.org/publishing-data

# | Domain | Phylum | Class | Order | Family | Genus | Species |
# | ------ | ------ | ----- | ----- | ------ | ----- | ------- |
# | 域     | 门     | 纲     | 目    | 科     | 属    | 种      |

# -----
# install.packages("\")
# # install.packages("dplyr")
# # install.packages("tidyr")

# install.packages("vegan")


In [ ]:
# NaaVRE parameters
# =====

# base settings
# -----
# dir_code <- "/home/jovyan/Virtual Labs/Open Lab/Git public"  # NaaVRE
dir_code <- file.path("/", "home", "jovyan")  # library: --volume="//c/DockerShare/ANERIS_DNA:/home/jovyan" naavre-fl-aneris-dna-jupyter:local
if (!dir.exists(dir_code)) {dir.create(dir_code, recursive=TRUE)}

dir_data <- file.path("/", "home", "jovyan", "Cloud Storage", "naa-vre-user-data")  # NaaVRE
if (!dir.exists(dir_data)) {dir.create(dir_data, recursive=TRUE)}

# workflow
# -----
# # 99
# param_gene_sequences = "SRR3231900,SRR3231901"
# param_workflow_id    = "wid-test_dataset_99-PEMA_Source_code_v.2.1.4"
# case_id              = "cid-test_dataset_99-PEMA_Source_code_v.2.1.4"
# # 00
# param_gene_sequences = "ERR7125480,ERR7125483,ERR7125486,ERR7125489"
# param_workflow_id    = "wid-test_dataset_00-WRiMS_Invasive_Checker"
# case_id              = "cid-test_dataset_00-WRiMS_Invasive_Checker"
# 01
param_gene_sequences = "ERR12541385,ERR12541386,ERR12541387,ERR7125481"
param_workflow_id    = "wid-test_dataset_01-PEMA_SequenceRetriever"
case_id              = "cid-test_dataset_01-PEMA_SequenceRetriever"
# # 02
# param_gene_sequences = "ERR2181465,ERR2181466,ERR2181467"
# param_workflow_id    = "wid-test_dataset_02-PEMA_Runner-OTU"
# case_id              = "cid-test_dataset_02-PEMA_Runner-OTU"
# # 03
# param_gene_sequences = "ERR3460470,ERR4018451,ERR4018452"
# param_workflow_id    = "wid-test_dataset_03-PEMA_Converter"
# case_id              = "cid-test_dataset_03-PEMA_Converter"

conf_dir_code     <- file.path(dir_code)
conf_dir_data     <- file.path(dir_data)
conf_dir_data_wid <- file.path(dir_data, param_workflow_id)
if (!dir.exists(conf_dir_data_wid)) {dir.create(conf_dir_data_wid, recursive=TRUE)}

# OTU
# -----
# pema_otu_delimiter = "\t"
# bold_otu_delimiter = ","
conf_delimiter_tsv = "\t"
conf_delimiter_csv = ","

print("Finish: NaaVRE parameters")
print(paste("Workspace", conf_dir_data_wid, sep=": "))


[1] "Finish: NaaVRE parameters"
[1] "Workspace: //home/jovyan/Cloud Storage/naa-vre-user-data/wid-test_dataset_99-PEMA_Source_code_v.2.1.4"


In [109]:
# PEMA-Converter
# =====

library(vegan)

case_name <- paste("PEMA", case_id, sep="-")
# print(paste("DevOps: case_name", case_name, sep=": "))

dir_otu <- file.path(conf_dir_data_wid, "OTU", case_name)
# print(paste("DevOps: dir_otu", dir_otu, sep=": "))

dir_pema <- file.path(conf_dir_data_wid, "PEMA-Converter", case_name)
# print(paste("DevOps: dir_pema", dir_pema, sep=": "))
if (!dir.exists(dir_pema)) {dir.create(dir_pema, recursive=TRUE)}

# -----
file_i_otu                <- file.path(dir_otu,  'otu.tsv')
file_o_abundance          <- file.path(dir_pema, 'abundance.csv')
file_o_aggregation        <- file.path(dir_pema, 'aggregation.csv')
file_o_abundance_no_dupls <- file.path(dir_pema, 'abundance_mds.csv')  # aggregation_no_dupls.csv
# print(paste("DevOps: file_i_otu", file_i_otu, sep=": "))

# import the biotic data, i.e. the final_table.tsv as it derives from PEMA
df_abundance <- read.csv(file_i_otu, sep=conf_delimiter_tsv, header=TRUE) 

# now we have an abundance table
# in the 1st column we have an OTU/ASV code
# then we have several columns, one for each sample
# and last, we have the classification column, with the taxonomy of each OTU/ASV

# start creating the aggregation file, needed for certain function (e.g. tax2dist)
df_taxonomy <- select(df_abundance, classification)
# print("DevOps: df_taxonomy-00")
# print(df_taxonomy)

# remove classification column from biotic data
df_abundance <- select(df_abundance, -classification)
# print("DevOps: df_abundance-00")
# print(df_abundance)

# save the biotic data, i.e. the community data that are needed as an input to 
# most of the RvLab functions
write.csv(df_abundance, file_o_abundance, quote=F, row.names=FALSE)

# now resume creating the aggregation file
# 域”、“门”、“纲”、“目”、“科”、“属”、“种”
colnames <- c("Domain", "Phylum", "Class", "Order", "Family", "Genus", "Species")
df_taxonomy <- separate(df_taxonomy, classification, colnames, sep = ";", remove = TRUE, convert = FALSE, extra = "warn", fill = "warn")
df_taxonomy <- cbind(df_abundance$OTU, df_taxonomy)
# print("DevOps: df_taxonomy-01")
# print(df_taxonomy)

names(df_taxonomy)[names(df_taxonomy) == "df_abundance$OTU"] <- "OTU"
# print("DevOps: df_taxonomy-02")
# print(df_taxonomy)

# check where there are NA values in the taxonomy data frame
colSums(is.na(df_taxonomy))

num_tries     = 0
previous_nans = -1

repeat {
    for (i in 1:nrow(df_taxonomy))
        if (is.na(df_taxonomy$Species[i])){
            df_taxonomy$Species[i] = df_taxonomy$Genus[i]
            df_taxonomy$Genus[i]   = df_taxonomy$Family[i]
            df_taxonomy$Family[i]  = df_taxonomy$Order[i]
            df_taxonomy$Order[i]   = df_taxonomy$Class[i]
            df_taxonomy$Class[i]   = df_taxonomy$Phylum[i]
    }

    current_nans = sum(is.na(df_taxonomy$Species))
    if (current_nans==0) {
        break
    }

    num_tries = num_tries + 1
    if (num_tries == 1000 && previous_nans == current_nans){
        stop("It seems that there are rows which cannot be fixed. Please check your data and contact the LifeWatch ERIC team for help.")
    }

    previous_nans = current_nans
}

# change the order of the columns
col_order <- c("OTU", "Species", "Genus", "Family", "Order", "Class", "Phylum", "Domain")
df_taxonomy_ordered <- df_taxonomy[, col_order]

# Remove duplicates based on Species column
df_taxonomy_ordered_no_dupls <- df_taxonomy_ordered[!duplicated(df_taxonomy_ordered$Species), ]

# save the aggregation file
write.csv(df_taxonomy_ordered_no_dupls, file_o_aggregation, quote=F, row.names=FALSE)

# create an abundance table that has the same rows as the aggregation file
df_abundance_no_dupls <- df_abundance %>% filter(OTU %in% df_taxonomy_ordered_no_dupls$OTU)

# save it
write.csv(df_abundance_no_dupls, file_o_abundance_no_dupls, quote=F, row.names=FALSE)

print("Finish: PEMA-Converter")
print(paste("Workspace", dir_pema, sep=": "))


Warning message:
“Expected 7 pieces. Additional pieces discarded in 4 rows [2, 5, 6, 8].”
Warning message:
“Expected 7 pieces. Missing pieces filled with `NA` in 5 rows [1, 3, 4, 7, 9].”


OTU  Domain  Phylum   Class   Order  Family   Genus Species 
      0       0       0       0       0       0       2       5

[1] "Finish: PEMA-Converter"
[1] "Workspace: //home/jovyan/Cloud Storage/naa-vre-user-data/wid-test_dataset_99-PEMA_Source_code_v.2.1.4/PEMA-Converter/PEMA-cid-test_dataset_99-PEMA_Source_code_v.2.1.4"


In [110]:
# metamds-observations
# =====
# In order to run this script, you need to have at least 2 samples in your abundance file

case_name <- paste("PEMA", case_id, sep="-")
# print(paste("DevOps: case_name", case_name, sep=": "))

dir_pema <- file.path(conf_dir_data_wid, "PEMA-Converter", case_name)
# print(paste("DevOps: dir_pema", dir_pema, sep=": "))

dir_metamds <- file.path(conf_dir_data_wid, "metamds-observations", case_name)
# print(paste("DevOps: dir_metamds", dir_metamds, sep=": "))
if (!dir.exists(dir_metamds)) {dir.create(dir_metamds, recursive=TRUE)}

fname_o_mds_fig <- file.path(dir_metamds, 'mds.png')

# -----
# args[1]: community matrix data file, i.e. abundance file
# file_abundance <- file.path(dir_pema, 'matrix_from_gitlab-dev.csv')
# file_abundance <- file.path(dir_pema, 'abundance.csv')
file_abundance <- file.path(dir_pema, 'abundance_mds.csv')

# args[2]: TRUE | FALSE (transpose matrix)
#   Note: dimension of df_matrix from file_abundance
#     Ture:  ncols >= 3
#     FALSE: nrows >= 3
is_transpose_abundance <- FALSE
# is_transpose_abundance <- TRUE

# args[3]: (Optional) factor file
#   Note: for mds plot, label & legend names, it's better not to provide
#     the rows should be the same as the names in the index of df_matrix, 
#     caution:
#        is_transpose_abundance
#        df_factor_col_label
# fname_factor_csv <- "factors_manual_from_gitlab-dev.csv"  # require col "AreaType"
fname_factor_csv <- "none"
if (fname_factor_csv != "none") {
    file_factor_csv <- file.path(dir_pema, fname_factor_csv)
} else {
    file_factor_csv <- "none"
}
# args[4]: if a factor file was used, the name of the column of the factor file that will be used as a label
# df_factor_col_label <- "LabelName"
df_factor_col_label <- "Location"
# df_factor_col_label <- "AreaType"

# args[5]: method for the calculation of distance matrix for the community matrix data file, e.g. euclidean, bray etc. 
mds_distance_matrix <- "bray"

# args[6]: TRUE | FALSE (autotransformation)
is_mds_auto_transformation <- FALSE

# args[7]: method for the calculation of pairwise distances, e.g. euclidean, bray etc.
mds_distance_pairwise <- "bray"

# args[8]: the number of permutations required for the permanova analysis
n_permutations <- 999

# import the community data matrix
df_matrix <- read.csv(file_abundance, sep=conf_delimiter_csv, header=TRUE, row.names=1)
print("DevOps: df_matrix-00")
print(df_matrix)

# replace the empty cells with zeros
df_matrix <-  df_matrix %>% replace(is.na(.), 0)

# import the factor file if it exists
if (file_factor_csv != "none") {
    df_factor <- read.csv(file_factor_csv, sep=conf_delimiter_csv, header=TRUE, row.names=1)
} else {
    # if not, create factors
    df_factor <- colnames(df_matrix)
}
print("DevOps: df_factor-00")
print(df_factor)

# transpose the community data matrix so that taxa would be the variables and samples the rows
if (is_transpose_abundance == TRUE)
    df_matrix <- t(df_matrix)

tmp_matrix_tot <- rowSums(df_matrix)
df_matrix <- df_matrix[tmp_matrix_tot > 0, ]
print("DevOps: df_matrix-01: rowSums(df_matrix) > 0")
print(df_matrix)

# create the mds
print("Info: Create the mds")
if (is_mds_auto_transformation == TRUE){
    sol_nmds <- metaMDS(df_matrix, distance=mds_distance_matrix, autotransform=TRUE)
}else{ 
    sol_nmds <- metaMDS(df_matrix, distance=mds_distance_matrix, autotransform=FALSE)
}

# store the stress value of the mds
print("Info: Store the stress value of the mds")
stress <- paste("Stress:", round(sol_nmds$stress, 2))
# create one label for each level of the chosen factor
if (file_factor_csv != "none") {
    # if(!df_factor_col_label %in% colnames(df_factor)) {
    #     df_factor$df_factor_col_label <- as.vector(rownames(df_factor))
    # }

    labels <- as.factor(df_factor[[df_factor_col_label]])
} else {
    # if not, create factors
    labels <- as.factor(df_factor)
}
# print("DevOps: labels-00")
# print(labels)

# create the mds plot
print("Info: Create the mds plot")
png(fname_o_mds_fig, height=800, width=1000, units="px")

op <- par(cex=1.5)
par(xpd=T, mar=par()$mar + c(0, 0, 0, 7))
plot(sol_nmds, type="n", ylab="", xlab="", yaxt="n", xaxt="n", bty="o") 
points(sol_nmds, col=labels, pch=16, cex=1.5)
text(x=1.4, y=2, labels=stress)
if (file_factor_csv != "none") {
    legend("bottomright", legend=unique(df_factor[[df_factor_col_label]]), col=unique(labels), pch=16, bty="n", inset=c(-0.17,0), xpd=T)
} else {
    # if not, create factors
    legend("bottomright", legend=unique(df_factor), col=unique(labels), pch=16, bty="n", inset=c(-0.17,0), xpd=T)
}
par(mar=c(5, 4, 4, 1) + 0.1)
dev.off()

# print the results of the function
print("Info: This is the result of the mds")
print(sol_nmds)

# run the permanova analysis
if (file_factor_csv != "none") {
    permanova <- adonis2(df_matrix ~ df_factor[[df_factor_col_label]], method=mds_distance_pairwise, data=df_factor, permutations=n_permutations)
    print("this is the result of the permanova");
    print(permanova)
} else {
    print("you cannot run the permanova analysis without a factor file")
    # permanova <- adonis2(df_matrix ~ df_factor, method=mds_distance_pairwise, data=df_factor)
    # print("this is the result of the permanova");
    # print(permanova)
}

print("Finish: metamds-observations")
print(paste("Workspace", dir_pema, sep=": "))


[1] "DevOps: df_matrix-00"
     SRR3231900 SRR3231901
Otu8         10          3
Otu9          3          2
Otu6         22         18
Otu4         43         31
Otu5         24         28
Otu2         46         13
Otu3         40         46
Otu1        127         89
[1] "DevOps: df_factor-00"
[1] "SRR3231900" "SRR3231901"
[1] "DevOps: df_matrix-01: rowSums(df_matrix) > 0"
     SRR3231900 SRR3231901
Otu8         10          3
Otu9          3          2
Otu6         22         18
Otu4         43         31
Otu5         24         28
Otu2         46         13
Otu3         40         46
Otu1        127         89
[1] "Info: Create the mds"
Run 0 stress 0.0002170693 
Run 1 stress 9.46095e-05 
... New best solution
... Procrustes: rmse 0.2087325  max resid 0.4290867 
Run 2 stress 9.949292e-05 
... Procrustes: rmse 0.01100429  max resid 0.01897858 
Run 3 stress 0.0007019882 
Run 4 stress 9.334725e-05 
... New best solution
... Procrustes: rmse 0.1973205  max resid 0.3275805 
Run 5 stress 

Warning message in metaMDS(df_matrix, distance = mds_distance_matrix, autotransform = FALSE):
“stress is (nearly) zero: you may have insufficient data”


[1] "Info: Store the stress value of the mds"
[1] "Info: Create the mds plot"


agg_record_1013092549 
                    2

[1] "Info: This is the result of the mds"

Call:
metaMDS(comm = df_matrix, distance = mds_distance_matrix, autotransform = FALSE) 

global Multidimensional Scaling using monoMDS

Data:     df_matrix 
Distance: bray 

Dimensions: 2 
Stress:     0 
Stress type 1, weak ties
Best solution was not repeated after 20 tries
The best solution was from try 15 (random start)
Scaling: centring, PC rotation, halfchange scaling 
Species: expanded scores based on ‘df_matrix’ 

[1] "you cannot run the permanova analysis without a factor file"
[1] "Finish: metamds-observations"
[1] "Workspace: //home/jovyan/Cloud Storage/naa-vre-user-data/wid-test_dataset_99-PEMA_Source_code_v.2.1.4/PEMA-Converter/PEMA-cid-test_dataset_99-PEMA_Source_code_v.2.1.4"
